# EXP-2026-001 on Colab: Stage 4 r4, start both streams from the pre-op pair

Sampling only, no training. This notebook loads the step-500 UNet that r3 saved (`stage4/r3/unet_step000500.pt` on Drive) and rebuilds the same 16 training slices. It then compares where sampling starts at t = 981. The arms were fixed in `configs/experiments/EXP-2026-001.json` (block `stage4_r4_pair_start`) before this run:

| Arm | Image stream starts from | Mask stream starts from |
|---|---|---|
| `pure_noise` | pure noise | pure noise (repeats r3) |
| `image_own_preop` | own pre-op image | pure noise (repeats r3) |
| `pair_own_preop` | own pre-op image | own pre-op mask (available at inference) |
| `pair_own_postop` | own post-op image | own post-op mask (oracle, never a result) |
| `pair_wrong_patient_preop` | another patient's pre-op image | that patient's pre-op mask (control) |

Noise, timestep, text and the concatenated pre-op conditions are the same in every arm. The first two arms must reproduce r3, which checks that the weights loaded correctly. Training data only; validation and test stay untouched. Expect about 12 minutes. Outputs go once to `synthesis/EXP-2026-001/stage4/<RUN_TAG>/` on Drive; the PNG grids show patient images, so keep them private.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "exp-2026-001-implementation"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
WORK = "/content/MedSegFactory"
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MSF_REPO, MSF_FILE = "JohnWeck/StableDiffusion", "checkpoint-300.pth"
RUN_TAG = "r4"  # use a new tag for every attempt; a run folder is never reused
WEIGHTS = "data/RHUH-GBM/synthesis/EXP-2026-001/stage4/r3/unet_step000500.pt"  # saved by r3

## 1. Mount Drive, get the latest code from the fork, link the data

In [ ]:
import copy
import json
import os
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
assert (PRIVATE / "split/split.json").exists(), "The synthesis folder is missing on Drive"
assert Path(WEIGHTS).exists(), f"{WEIGHTS} is missing: r3 must have run with SAVE_UNET_TO_DRIVE = True"
RUN = PRIVATE / "stage4" / RUN_TAG
assert not RUN.exists(), f"{RUN} exists: choose a new RUN_TAG (records are never overwritten)"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                "nibabel>=5.3,<6", "SimpleITK==2.5.6", "pydicom>=3,<4", "scikit-image>=0.25"], check=True)
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(commit)
STARTED = time.perf_counter()

## 2. Models: MedSegFactory checkpoint (strict), frozen VAE and CLIP, sampler

In [ ]:
import numpy as np
import torch
from diffusers import AutoencoderKL
from huggingface_hub import hf_hub_download
from transformers import CLIPTextModel, CLIPTokenizer

from rhuh_synthesis.manifest import file_sha256
from rhuh_synthesis.model import reference_ddim_scheduler
from rhuh_synthesis.provenance import code_version, config_sha256, environment
from rhuh_synthesis.stage3 import load_reference_unet
from StableDiffusion.Our_UNet import UNet2DConditionModel

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))
S4 = config["stage4_overfit"]
DIAG = config["stage4_r2_diagnostics"]  # metric definitions
R4 = config["stage4_r4_pair_start"]
checkpoint = hf_hub_download(MSF_REPO, MSF_FILE)
unet_config = UNet2DConditionModel.load_config(SD_REPO, subfolder="unet")
unet4, load_report = load_reference_unet(UNet2DConditionModel, unet_config, checkpoint)
assert unet4 is not None, "Checkpoint keys do not match the vendored UNet: stop and report"
vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae").to("cuda").eval().requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder").to("cuda").eval().requires_grad_(False)
ddim = reference_ddim_scheduler()
record = {
    "stage": "4_overfit_r4_pair_start", "run_tag": RUN_TAG, "commit": commit, "code_version": code_version(WORK),
    "config_sha256": config_sha256(config), "stage4_r4_config": R4, "environment": environment(),
    "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "checkpoint": {"repo": MSF_REPO, "file": MSF_FILE, "sha256": file_sha256(checkpoint),
                   "missing_keys": len(load_report["missing_keys"]), "unexpected_keys": len(load_report["unexpected_keys"])},
}
print({k: record[k] for k in ("commit", "config_sha256", "torch", "cuda", "gpu")}, record["checkpoint"])

## 3. Load the r3 step-500 weights (strict) into the 11-channel UNet

In [ ]:
from rhuh_synthesis.overfit import load_trained_unet

unet11, record["weights"] = load_trained_unet(unet4, WEIGHTS, device="cuda")
record["weights"]["path"] = WEIGHTS
del unet4
torch.cuda.empty_cache()
print(record["weights"])

## 4. Rebuild the 16 training slices and cache their features

In [ ]:
from rhuh_synthesis.overfit import label_volumes, ranked_patients, restrict, select_overfit_set
from rhuh_synthesis.representation import crop_forward
from rhuh_synthesis.stage3 import clip_encoder
from rhuh_synthesis.training import build_feature_cache
from rhuh_synthesis.working_grid import to_working_grid

split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
train_patients = [p for p in split["roles"]["train"]["patients"] if p not in policy["excluded_training"]]
paths = {p: next(PRIVATE.glob(f"export_v1/train*/{p}.npz")) for p in train_patients}
volumes = {}
for patient, path in paths.items():
    with np.load(path) as data:
        volumes[patient] = label_volumes(data["label"])
ranked = ranked_patients(volumes)


def load_grid(patient):
    with np.load(paths[patient]) as data:
        arrays = {key: data[key] for key in data.files}
    crop, grid = to_working_grid(arrays)
    grid["post_support"] = crop_forward(arrays["post_support"].astype(np.uint8), crop).astype(bool)
    return grid


selection, grids, skipped = select_overfit_set(ranked, load_grid, positive=S4["patients"]["positive"],
                                               negative=S4["patients"]["negative"], **S4["slices_per_patient"])
patients = [entry["patient"] for entry in selection]
exports = {entry["patient"]: restrict(grids[entry["patient"]], [z for z, _ in entry["slices"]]) for entry in selection}
items = [(entry["patient"], k) for entry in selection for k in range(len(entry["slices"]))]
categories = [category for entry in selection for _, category in entry["slices"]]
groups = [patient for patient, _ in items]
letters = {entry["patient"]: "ABCD"[n] for n, entry in enumerate(selection)}
names = [f"{letters[p]}-{c}-{k + 1}" for (p, k), c in zip(items, categories)]
targets = [{"post_image": exports[p]["post_image"][k], "pre_image": exports[p]["pre_image"][k],
            "label": exports[p]["label"][k], "region": exports[p]["pre_foreground"][k] & exports[p]["post_support"][k],
            "brain": exports[p]["pre_foreground"][k]} for p, k in items]
pixel_mm = config["representation"]["crop_mm"] / config["representation"]["out_px"]
del grids
cache = build_feature_cache(exports, vae, clip_encoder(tokenizer, text_encoder, "cuda"), config["text"],
                            batch=8, device="cuda", mask_codec=config["representation"]["mask_codec"])
record["selection"] = [{"letter": letters[e["patient"]], "patient": e["patient"], "kind": e["kind"],
                        "rank": ranked[e["kind"]].index(e["patient"]) + 1,
                        "slices": [{"frame_z": z, "category": c} for z, c in e["slices"]]} for e in selection]
record["skipped"] = [dict(s, rank=ranked[s["kind"]].index(s["patient"]) + 1) for s in skipped]
slice_table = [{"name": n, "label_px": {f"id{c}": int((t["label"] == c).sum()) for c in (1, 2, 3)},
                "pre_label_px": int((exports[p]["pre_label"][k] > 0).sum())} for n, t, (p, k) in zip(names, targets, items)]
record["slice_table"] = slice_table
print("skipped (rank, kind, reason):", [(s["rank"], s["kind"], s["reason"]) for s in record["skipped"]])
for row in slice_table:
    print(row)

## 5. The five starting arms (the post-op pair is an oracle, never a result)

In [ ]:
from rhuh_synthesis.overfit import (generate, image_start_latents, mask_latents, pair_start_readings,
                                   save_overfit_grid, score_slices, shifted_conditions, summarize)

GENERATION = dict(num_inference_steps=config["sampling"]["num_inference_steps"],
                  guidance_scale=config["sampling"]["guidance_scale"], seed=S4["generation"]["seed"],
                  batch=S4["generation"]["batch_pairs"], mask_codec=config["representation"]["mask_codec"])


def scored(images, masks, name):
    rows = score_slices(images, masks, targets, pixel_mm=pixel_mm,
                        foreground_threshold=DIAG["metrics"]["foreground_threshold"])
    panels = [{"name": n, "pre_image": t["pre_image"], "post_image": t["post_image"], "generated_image": i,
               "label": t["label"], "generated_mask": m} for n, t, i, m in zip(names, targets, images, masks)]
    save_overfit_grid(RUN / "grids" / f"{name}.png", panels, title=f"Stage 4 {RUN_TAG} {name}")
    return {"slices": [dict(row, name=n) for row, n in zip(rows, names)],
            "summary": summarize(rows, categories, groups=groups)}


RUN.mkdir(parents=True)
started = time.perf_counter()
index = {item: n for n, item in enumerate(items)}
wrong = [index[item] for item in shifted_conditions(items, patients)]
pre_image, post_image = image_start_latents(cache, items, "pre"), image_start_latents(cache, items, "post")
pre_mask = mask_latents(vae, np.stack([exports[p]["pre_label"][k] for p, k in items]),
                        mask_codec=config["representation"]["mask_codec"], device="cuda")
post_mask = torch.stack([cache["patients"][p]["mask_mean"][k] for p, k in items]) * cache["scaling_factor"]
ARMS = {
    "pure_noise": {},
    "image_own_preop": {"start_image": pre_image},
    "pair_own_preop": {"start_image": pre_image, "start_mask": pre_mask},
    "pair_own_postop": {"start_image": post_image, "start_mask": post_mask},
    "pair_wrong_patient_preop": {"start_image": pre_image[wrong], "start_mask": pre_mask[wrong]},
}
arms = {}
for name, starts in ARMS.items():
    arms[name] = scored(*generate(unet11, vae, ddim, cache, items, **starts, **GENERATION), name)
    print(f"{name}:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in arms[name]["summary"].items()})
readings = pair_start_readings(arms, R4["reading"])
record["arms_seconds"] = time.perf_counter() - started
print(json.dumps(readings, indent=1))
print(f"arms took {record['arms_seconds']:.0f}s")

## 6. Write-once record and the part to send

In [ ]:
from rhuh_synthesis.manifest import write_once_json

record.update(arms=arms, readings=readings, total_seconds=time.perf_counter() - STARTED,
              peak_gib=torch.cuda.max_memory_allocated() / 2**30)
path, digest = write_once_json(RUN / "record.json", record)
print("saved", path, digest[:12])
shareable = {
    "commit": commit, "config_sha256": record["config_sha256"][:12], "gpu": record["gpu"],
    "weights": {k: (v[:12] if k == "sha256" else v) for k, v in record["weights"].items() if k != "path"},
    "skipped": [(s["rank"], s["kind"], s["reason"]) for s in record["skipped"]],
    "slices": slice_table,
    "arms": {name: {"summary": arm["summary"], "slices": arm["slices"]} for name, arm in arms.items()},
    "readings": readings, "arms_seconds": round(record["arms_seconds"]), "peak_gib": round(record["peak_gib"], 1),
}
print(json.dumps(shareable, indent=1))

## Next

Save this notebook with its outputs under a name Git ignores (for example `RHUH_GBM_Colab_stage4_r4ipynb.ipynb`). Then download `record.json` and the PNG grids from `synthesis/EXP-2026-001/stage4/<RUN_TAG>/` on Drive into the same folder on your laptop.

If the verdict is `preop_pair_start_justified`, the pre-op-pair start is declared as a Stage 5 sampling arm, alongside pure noise. Stage 5 still needs your go-ahead.